## Step 1: Assemble Corpus

In [2]:
import re
import requests
from collections import Counter, defaultdict

import numpy as np
import time


In [3]:

DATASET_URL = "https://raw.githubusercontent.com/bhagatpandey369/NLP/refs/heads/main/dataset/1-18%20books%20combined.txt"
CORPUS_FILE = "mahabharata_corpus.txt"

In [4]:
response = requests.get(DATASET_URL)

response.raise_for_status()
corpus = response.text
print(corpus[:200])

Adi Parva
 
Chapter One
Maharaja Shantanu Marries the Celestial Ganga
 
According to the historical records of this earth, there once lived a King named Maharaja Shantanu, the son of Pratipa, who took


In [5]:
len(corpus.splitlines())

3118

In [6]:
len(corpus.split())

290922

In [7]:
len(corpus)

1701795

In [8]:
text = corpus.lower()
tokens = re.findall(r'\b\w+\b', text)
len(tokens)

293626

In [9]:
text[:100]

'adi parva\n \nchapter one\nmaharaja shantanu marries the celestial ganga\n \naccording to the historical '

In [10]:
tokens[:10]

['adi',
 'parva',
 'chapter',
 'one',
 'maharaja',
 'shantanu',
 'marries',
 'the',
 'celestial',
 'ganga']

In [11]:
word_counts = Counter(tokens)
len(word_counts)

10335

In [12]:
for word, count in word_counts.most_common(20):
    print(word, '-->', count)

the --> 20561
of --> 10737
and --> 9651
to --> 7874
in --> 4369
his --> 3962
a --> 3469
he --> 3294
with --> 3182
that --> 2976
you --> 2925
is --> 2785
i --> 2632
was --> 2545
by --> 2308
this --> 2211
s --> 2127
all --> 1881
arjuna --> 1881
for --> 1858


## Step 2 — Define Vocabulary Size (M)

In [13]:
M = 1000

# reserve one slot in the vocabulary for an <unk> bucket
most_common_words = word_counts.most_common(M - 1)
most_common_words[-10:]


[('princess', 34),
 ('blind', 34),
 ('assist', 34),
 ('aside', 34),
 ('reactions', 34),
 ('attachment', 34),
 ('meant', 34),
 ('conchshells', 34),
 ('witnessing', 34),
 ('pilgrimage', 34)]

In [14]:
UNK_TOKEN = "<unk>"

vocab = [word for word, count in most_common_words] + [UNK_TOKEN]
vocab[:10]


['the', 'of', 'and', 'to', 'in', 'his', 'a', 'he', 'with', 'that']

In [15]:
len(vocab)

1000

In [16]:
word_to_index = {word: idx for idx, word in enumerate(vocab)}
UNK_ID = word_to_index[UNK_TOKEN]


In [17]:
for word in vocab[:10]:
    print(word, '-->', word_to_index[word])

the --> 0
of --> 1
and --> 2
to --> 3
in --> 4
his --> 5
a --> 6
he --> 7
with --> 8
that --> 9


In [18]:
idx_to_word = {idx: word for word, idx in word_to_index.items()}
for idx in range(10):
    print(idx, '-->', idx_to_word[idx])

0 --> the
1 --> of
2 --> and
3 --> to
4 --> in
5 --> his
6 --> a
7 --> he
8 --> with
9 --> that


In [19]:
len(tokens)

293626

In [20]:
corpus_ids = np.array(
    [word_to_index.get(word, UNK_ID) for word in tokens],
    dtype=np.int32
)


In [21]:
len(corpus_ids)

293626

In [22]:
corpus_ids

array([999, 354, 148, ..., 660, 417, 417], dtype=int32)

In [23]:
unk_fraction = np.mean(corpus_ids == UNK_ID)
round((1 - unk_fraction) * 100, 2)


np.float64(84.02)

In [24]:
for idx in corpus_ids[:10]:
    print(idx_to_word[idx], end=" ")

<unk> parva chapter one maharaja shantanu <unk> the celestial ganga 

## Step 3 — Choose Context Window (C)

In [25]:
# Context window

C = 2

window_size= 2 * C + 1
window_size

5

In [26]:
example_words = [idx_to_word[idx] for idx in corpus_ids[:10]]
' '.join(example_words)

'<unk> parva chapter one maharaja shantanu <unk> the celestial ganga'

In [27]:
for i in range(C, C+3):
    target_id = corpus_ids[i]
    target_word = idx_to_word[target_id]

    context = []

    for j in range(i-C, i + C +1):
        if j != i:
            context.append(idx_to_word[corpus_ids[j]])

    print(target_word, '-->', context)    

chapter --> ['<unk>', 'parva', 'one', 'maharaja']
one --> ['parva', 'chapter', 'maharaja', 'shantanu']
maharaja --> ['chapter', 'one', 'shantanu', '<unk>']


## Step 4: Build Co-occurrence Dictionary

In [28]:
cooccurrence = defaultdict(Counter)
cooccurrence

defaultdict(collections.Counter, {})

In [29]:
for i in range(C, len(corpus_ids) - C):

    traget = int(corpus_ids[i])

    for j in range(i-C, i+C+1):
        if i == j:
            continue

        context = int(corpus_ids[j])
        cooccurrence[traget][context] += 1

In [30]:
cooccurrence[100].most_common(5)

[(999, 147), (0, 141), (5, 135), (2, 115), (445, 71)]

In [31]:
for traget_id in range(5):
    traget_word = idx_to_word[traget_id]

    context_words = [(idx_to_word[context_id], count) for context_id, count in cooccurrence[traget_id].most_common(5)]

    print(traget_word, '-->', context_words)

the --> [('<unk>', 16719), ('of', 8165), ('and', 2423), ('to', 2270), ('in', 1658)]
of --> [('the', 8165), ('<unk>', 8114), ('and', 849), ('son', 676), ('a', 607)]
and --> [('<unk>', 8237), ('the', 2423), ('his', 849), ('of', 849), ('to', 627)]
to --> [('<unk>', 5720), ('the', 2270), ('and', 627), ('be', 554), ('his', 535)]
in --> [('<unk>', 3592), ('the', 1658), ('this', 376), ('and', 335), ('battle', 277)]


In [32]:
def show_cooccurrences(word, tok_k):

    if word not in word_to_index:
        return "Word not found in vocabulary."

    target_id = word_to_index[word]

    print('Target word', word)
    print('-'*10)

    for context_id, count in cooccurrence[target_id].most_common(tok_k):
        print(idx_to_word[context_id],'-->', count)


In [33]:
show_cooccurrences('krishna', 5)

Target word krishna
----------
lord --> 907
<unk> --> 625
and --> 246
the --> 236
of --> 186


In [34]:
show_cooccurrences('apple', 5)

'Word not found in vocabulary.'

## Step 5: Choose Embedding Size (N)

In [35]:
# Embedding Dimension
N = 100

print('Embedding Matrix:')
M , '*' , N

Embedding Matrix:


(1000, '*', 100)

## Step 6: Initialize Two Tables (E: Target words and U: Context Word)

In [36]:
np.random.seed(42)

# random initialization
E = np.random.normal(loc=0.0, scale=0.01, size=(M, N)).astype(np.float32)
U = np.random.normal(loc=0.0, scale=0.01, size=(M, N)).astype(np.float32)


In [37]:
E.shape

(1000, 100)

In [38]:
U.shape

(1000, 100)

In [39]:
word = vocab[100]
word_id = word_to_index[word]
print('word ', word)
print('ID ', word_id)
print('Embedding word ', E[word_id][:5])

word  bow
ID  100
Embedding word  [-0.00678495 -0.00305499 -0.00597381  0.00110418  0.01197179]


## Step 7 — Train Embeddings with SGNS

In [40]:
learning_rate = 0.025
negative_samples = 5
epoch = 5

In [41]:
covered_count = sum(count for _, count in most_common_words)
unk_count = len(tokens) - covered_count

# vocab is [top M-1 words..., <unk>], so align counts the same way
vocab_counts = np.array(
    [count for _, count in most_common_words] + [unk_count],
    dtype=np.float64
)
vocab_counts[:20]


array([20561., 10737.,  9651.,  7874.,  4369.,  3962.,  3469.,  3294.,
        3182.,  2976.,  2925.,  2785.,  2632.,  2545.,  2308.,  2211.,
        2127.,  1881.,  1881.,  1858.])

In [42]:
negative_distribution = vocab_counts ** 0.75
negative_distribution = negative_distribution / negative_distribution.sum()
negative_distribution[:20]

array([0.0326489 , 0.02005615, 0.01851461, 0.01589396, 0.01021814,
       0.00949558, 0.00859486, 0.00826757, 0.00805583, 0.00766144,
       0.00756275, 0.00728961, 0.00698715, 0.0068132 , 0.00633158,
       0.00613094, 0.0059554 , 0.00543097, 0.00543097, 0.00538108])

In [43]:
negative_distribution.sum()

np.float64(1.0)

In [44]:
def sigmoid(x):
    x = np.clip(x, -15, 15)
    x = 1.0 / (1.0 + np.exp(-x))
    return x

In [45]:
values = np.array([-5, -2, 0, 2, 5])
sigmoid(values)

array([0.00669285, 0.11920292, 0.5       , 0.88079708, 0.99330715])

In [46]:
len(corpus_ids)

293626

In [47]:
def generate_context_pairs(corpus_ids, window_size):
    n = len(corpus_ids)

    for i in range(n):

        target = int(corpus_ids[i])

        start = max(0, i - window_size)
        end = min(n, i + window_size + 1)

        for j in range(start, end):

            if i == j:
                continue

            context = int(corpus_ids[j])

            yield target, context

In [48]:
pair_generator = generate_context_pairs(corpus_ids[:10], C)

for target, context in list(pair_generator)[:10]:

    print(f"{idx_to_word[target]:15s} -> "f"{idx_to_word[context]}")

<unk>           -> parva
<unk>           -> chapter
parva           -> <unk>
parva           -> chapter
parva           -> one
chapter         -> <unk>
chapter         -> parva
chapter         -> one
chapter         -> maharaja
one             -> parva


In [49]:
def train_sgns(corpus_ids, E, U, negative_distribution,
               window_size=2, negative_samples=5, learning_rate=0.025,
               epochs=3, max_tokens=None):
    if max_tokens is not None:
        train_corpus = corpus_ids[:max_tokens]

    else:
        train_corpus = corpus_ids

    vocab_size = len(negative_distribution)

    for epoch in range(epochs):
        start_time = time.time()

        total_loss = 0.0
        pair_count = 0

        n = len(train_corpus)

        for i in range(n):
            target_id = int(train_corpus[i])

            start = max(0, i - window_size)
            end = min(n, i + window_size + 1)

            target_vector = E[target_id].copy()

            for j in range(start, end):
                if i == j:
                    continue

                context_id = int(train_corpus[j])

                # positive sample
                context_vector = U[context_id].copy()
                score = np.dot(target_vector, context_vector)
                prediction = sigmoid(score)
                error = prediction - 1
                total_loss = total_loss - np.log(max(prediction, 1e-10))

                grad_target = error * context_vector
                grad_context = error * target_vector

                E[target_id] = E[target_id] - learning_rate * grad_target
                U[context_id] = U[context_id] - learning_rate * grad_context

                # negative samples: resample on collision so we always end up
                # with exactly `negative_samples` valid negatives, and make
                # sure a "negative" is never the target word itself either.
                negative_ids = []
                while len(negative_ids) < negative_samples:
                    candidate = int(np.random.choice(vocab_size, p=negative_distribution))
                    if candidate != context_id and candidate != target_id:
                        negative_ids.append(candidate)

                for negative_id in negative_ids:
                    negative_vector = U[negative_id].copy()

                    negative_score = np.dot(target_vector, negative_vector)
                    negative_prediction = sigmoid(negative_score)

                    negative_error = negative_prediction

                    total_loss = total_loss - np.log(max(1 - negative_prediction, 1e-10))

                    grad_target_negative = negative_error * negative_vector
                    grad_negative = negative_error * target_vector

                    E[target_id] = E[target_id] - (learning_rate * grad_target_negative)
                    U[negative_id] = U[negative_id] - (learning_rate * grad_negative)

                pair_count = pair_count + 1

        total_time = time.time() - start_time
        average_loss = total_loss / max(pair_count, 1)
        print(f"Epoch {epoch + 1}/{epochs} | "f"Pairs: {pair_count:,} | "f"Loss: {average_loss:.4f} | "f"Time: {total_time:.2f}s")

    return E, U


In [50]:
E, U = train_sgns(
    corpus_ids=corpus_ids,
    E=E,
    U=U,
    negative_distribution=negative_distribution,
    window_size=C,
    negative_samples=negative_samples,
    learning_rate=learning_rate,
    epochs=2,
    max_tokens= None
)

Epoch 1/2 | Pairs: 1,174,498 | Loss: 2.2943 | Time: 431.31s
Epoch 2/2 | Pairs: 1,174,498 | Loss: 2.1657 | Time: 431.57s


In [51]:
E.shape

(1000, 100)

In [52]:
word = 'arjuna'
if word in word_to_index:
    word_index = word_to_index[word]
    print("Word ID:", word_index)
    print("\nLearned embedding:")
    print(E[word_index][:5])


Word ID: 18

Learned embedding:
[-0.15729977  0.14926147 -0.11887599 -0.38971764 -0.13130398]


In [53]:
def cosine_similarity(vec1, vec2):

    denominator = (np.linalg.norm(vec1) * np.linalg.norm(vec2))

    if denominator == 0:
        return 0.0
    return np.dot(vec1, vec2) / denominator

In [54]:
def most_similar(word, E, word_to_index, idx_to_word, top_k=10):

    if word not in word_to_index:
        return 'Word not found in vocabulary.'

    word_id  = word_to_index[word]
    traget_vector = E[word_id]

    norms = np.linalg.norm(E, axis=1, keepdims=True)
    normalized_E = E / np.maximum(norms, 1e-10)

    target_normalized = (traget_vector / max(np.linalg.norm(traget_vector), 1e-10))

    similarities = normalized_E @ target_normalized
    similarities[word_id] = -np.inf

    top_indices = np.argsort(similarities)[-top_k:][::-1]
    print(f"\nWords most similar to '{word}':\n")

    for idx in top_indices:
        print(
            f"{idx_to_word[idx]:20s} "
            f"{similarities[idx]:.4f}"
        )

In [55]:
most_similar("arjuna", E, word_to_index, idx_to_word)


Words most similar to 'arjuna':

bhima                0.7117
uttara               0.7020
kichaka              0.7004
parashurama          0.6887
kuvera               0.6864
shalva               0.6638
satyaki              0.6531
vyasadeva            0.6459
bhimasena            0.6449
ghatotkacha          0.6418


In [56]:
most_similar("king", E, word_to_index, idx_to_word)


Words most similar to 'king':

daughter             0.6928
princess             0.6770
monarch              0.6639
maharaja             0.6357
brahmana             0.6343
narada               0.6323
queen                0.6153
prince               0.6096
gandhari             0.6093
suta                 0.5985


In [57]:
most_similar("war", E, word_to_index, idx_to_word)


Words most similar to 'war':

formation            0.8019
future               0.7777
young                0.7770
demon                0.7560
river                0.7453
prince               0.7362
lake                 0.7357
distance             0.7339
ascetic              0.7332
ceremony             0.7288
